In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pickle

In [12]:
with open("training_data.txt", "r", encoding="utf-8") as f :
    text = f.read()

print('characters : ', len(text))
print()
print(text[:300])

characters :  1420167

BOOK I.
The quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mother Thetis to ask Jove to help the Trojans—Scene between Jove and Juno on Olympus.

Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans. Many a brave s


In [13]:
with open("bpe_tokenizer.pkl", "rb") as f:
    tokenizer_data = pickle.load(f)

merges = tokenizer_data["merges"]
bpe_stoi = tokenizer_data["bpe_stoi"]
bpe_itos = tokenizer_data["bpe_itos"]
bpe_vocab = tokenizer_data["bpe_vocab"]
bpe_vocab_size = tokenizer_data["bpe_vocab_size"]

print("Number of merges:", len(merges))
print("Vocabulary size:", bpe_vocab_size)

Number of merges: 1000
Vocabulary size: 1080


In [14]:
def merge_pair(tokens, pair, new_token):
    new_tokens = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(new_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return new_tokens


def bpe_encode(text):
    tokens = list(text)

    for pair, new_token in merges.items():
        tokens = merge_pair(tokens, pair, new_token)

    return [bpe_stoi[token] for token in tokens]


def bpe_decode(token_ids):
    return "".join(bpe_itos[token_id] for token_id in token_ids)

In [15]:
sample_text = "The quarrel between Agamemnon and Achilles."

encoded = bpe_encode(sample_text)
decoded = bpe_decode(encoded)

print("Original:", sample_text)
print("Encoded:", encoded)
print("Decoded:", decoded)
print("Exact match:", sample_text == decoded)

Original: The quarrel between Agamemnon and Achilles.
Encoded: [789, 364, 100, 66, 855, 162, 321, 818, 922, 92, 407, 118, 7]
Decoded: The quarrel between Agamemnon and Achilles.
Exact match: True


In [16]:
print("Total characters:", len(text))
print("First 100 characters:", repr(text[:100]))
print("Last 100 characters:", repr(text[-100:]))

Total characters: 1420167
First 100 characters: 'BOOK I.\nThe quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mo'
Last 100 characters: 'form and voice of Mentor, and presently made a covenant of peace between the two contending parties.'


In [17]:
bpe_encoded = bpe_encode(text)

bpe_data = torch.tensor(bpe_encoded, dtype=torch.long)

n = int(0.9 * len(bpe_data))

bpe_train_data = bpe_data[:n]
bpe_val_data = bpe_data[n:]

print("Total BPE tokens:", len(bpe_data))
print("Training tokens:", len(bpe_train_data))
print("Validation tokens:", len(bpe_val_data))
print("BPE vocabulary size:", bpe_vocab_size)

Total BPE tokens: 424914
Training tokens: 382422
Validation tokens: 42492
BPE vocabulary size: 1080


In [18]:
def get_bpe_batch(split, block_size=64, batch_size=4):
    data = bpe_train_data if split == "train" else bpe_val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])

    return x, y

In [19]:
xb, yb = get_bpe_batch("train")

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("\nInput tokens:", xb[0])
print("Target tokens:", yb[0])

print("\nDecoded input:")
print(bpe_decode(xb[0].tolist()))

print("\nDecoded target:")
print(bpe_decode(yb[0].tolist()))

Input shape: torch.Size([4, 64])
Target shape: torch.Size([4, 64])

Input tokens: tensor([ 791,  254,  162,   52,  976,  502,   88, 1036,  119,  282,  326,  102,
          51,  177,   28,  128,  112,  117,   54,  304,  260,  120,  126,  993,
         586,  106,  606,  316,  155,  570,  119,   51,  496,  988,  423,  296,
         537,  113,  439,   89,  217,  109,   98,  155,  581,  570,  155,  227,
         142,  590,   52,  281,   83,   71,  164,  290,  144,  260,  252,  282,
         163,  573,  122,  996])
Target tokens: tensor([ 254,  162,   52,  976,  502,   88, 1036,  119,  282,  326,  102,   51,
         177,   28,  128,  112,  117,   54,  304,  260,  120,  126,  993,  586,
         106,  606,  316,  155,  570,  119,   51,  496,  988,  423,  296,  537,
         113,  439,   89,  217,  109,   98,  155,  581,  570,  155,  227,  142,
         590,   52,  281,   83,   71,  164,  290,  144,  260,  252,  282,  163,
         573,  122,  996,  386])

Decoded input:
to his own bedchamber

In [20]:
class SelfAttentionHead(nn.Module):
    def __init__(self, head_size, embedding_dim, block_size):
        super().__init__()

        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.shape[-1] ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)

        v = self.value(x)

        out = wei @ v

        return out

In [21]:
embedding_dim = 32
num_heads = 4
block_size = 64
batch_size = 4

head_size = embedding_dim // num_heads

test_head = SelfAttentionHead(
    head_size=head_size,
    embedding_dim=embedding_dim,
    block_size=block_size
)

test_input = torch.randn(batch_size, block_size, embedding_dim)

test_output = test_head(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", test_output.shape)
print("Head size:", head_size)

Input shape: torch.Size([4, 64, 32])
Output shape: torch.Size([4, 64, 8])
Head size: 8


In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, embedding_dim, block_size):
        super().__init__()

        self.heads = nn.ModuleList([
            SelfAttentionHead(head_size, embedding_dim, block_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(num_heads * head_size, embedding_dim)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)

        return self.proj(out)

In [23]:
test_multi_head = MultiHeadAttention(
    num_heads=num_heads,
    head_size=head_size,
    embedding_dim=embedding_dim,
    block_size=block_size
)

multi_head_output = test_multi_head(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", multi_head_output.shape)
print("Number of attention heads:", num_heads)
print("Combined head dimensions:", num_heads * head_size)

Input shape: torch.Size([4, 64, 32])
Output shape: torch.Size([4, 64, 32])
Number of attention heads: 4
Combined head dimensions: 32


In [24]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )

    def forward(self, x):
        return self.net(x)

In [25]:
test_feed_forward = FeedForward(embedding_dim)

feed_forward_output = test_feed_forward(multi_head_output)

print("Input shape:", multi_head_output.shape)
print("Output shape:", feed_forward_output.shape)

Input shape: torch.Size([4, 64, 32])
Output shape: torch.Size([4, 64, 32])


In [26]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, block_size):
        super().__init__()

        head_size = embedding_dim // num_heads

        self.sa = MultiHeadAttention(
            num_heads,
            head_size,
            embedding_dim,
            block_size
        )

        self.ffwd = FeedForward(embedding_dim)

        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size, num_heads, num_layers):
        super().__init__()

        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding_table = nn.Embedding(block_size, embedding_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, block_size)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)

        self.lm_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        token_embeddings = self.token_embedding_table(idx)

        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)

        x = token_embeddings + position_embeddings

        x = self.blocks(x)
        x = self.ln_final(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(
                logits.reshape(B * T, -1),
                targets.reshape(B * T)
            )

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx